Load config

In [0]:
%run ../config/config

In [0]:
pipeline_table=f"{catalog}.{pipeline_schema}.pipeline_control"

Identify batches, unprocessed batches, and next batch

In [0]:
class IdentifyBatches():
    """
    Class to identify batches to process.
    
    """

    def __init__(self, landing_zone_path, pipeline_table):
        self.landing_zone_path=landing_zone_path
        self.pipeline_table=pipeline_table

    def get_folders(self) -> list[str]:
        """
        Returns a list of folders in the landing zone.
        """
        batch_folders=sorted([file.name.rstrip("/")for file in dbutils.fs.ls(landing_zone_path)if file.isDir()])

        return batch_folders
    
    def get_tracked_batches(self) -> list[str]:
        """
        Returns a list of tracked batches.
        """
        from pyspark.sql import functions as F

        if spark.catalog.tableExists(self.pipeline_table):
            self.tracked_batches = [
                row.batch_id
                for row in (
                    spark.table(self.pipeline_table)
                        .filter(F.col("status").isin("in_progress", "completed"))
                        .select("batch_id")
                        .distinct()
                        .collect()
                )
            ]
        else:
            self.tracked_batches = []

        return self.tracked_batches
    
    def get_next_batch(self) -> str:
        """
        Returns the next batch to process.
        """
        new_batches = sorted(list(set(self.get_folders()) - set(self.get_tracked_batches())))
        next_batch = new_batches[0] if new_batches else None
        
        return next_batch

In [0]:
batches=IdentifyBatches(landing_zone_path,pipeline_table)
list_batches=batches.get_folders()
tracked_batches=batches.get_tracked_batches()
next_batch=batches.get_next_batch()

print(f"Landing batches: {list_batches}")
print(f"Tracked batches: {tracked_batches}")
print(f"Next to process: {next_batch}")

if next_batch is None:
    dbutils.jobs.taskValues.set(key="batch_id", value="")
    dbutils.jobs.taskValues.set(key="has_batch", value="false")
else:
    dbutils.jobs.taskValues.set(key="batch_id", value=next_batch)
    dbutils.jobs.taskValues.set(key="has_batch", value="true")

Landing batches: ['2022-1', '2022-2']
Tracked batches: ['2022-1']
Next to process: 2022-2
